In [ ]:
import os
import numpy as np
import pandas as pd
from typing import Tuple, Optional

In [ ]:
SRC_XLSX = "data/processed/bmi_first_threshold.xlsx"
OUT_DIR  = ""

W_MIN, W_MAX = 10.0, 25.0
C_FAIL = 1.0
C_WAIT = 0.04

N_BOOT = 500
STEP   = 0.1

def to_weeks(first_time: pd.Series) -> pd.Series:
    s = pd.to_numeric(first_time, errors="coerce")
    return s/7.0 if s.median() > 40 else s

def empirical_success_prob(weeks_array: np.ndarray, w: float) -> float:
    if len(weeks_array) == 0:
        return np.nan
    return float(np.mean(weeks_array <= w))

def risk(w: float, weeks_array: np.ndarray) -> float:
    p = empirical_success_prob(weeks_array, w)
    if np.isnan(p):
        return np.inf
    return C_FAIL*(1.0 - p) + C_WAIT*max(0.0, w - W_MIN)

def find_optimal_week(weeks_array: np.ndarray,
                      w_min=W_MIN, w_max=W_MAX, step=STEP) -> Tuple[float, float, float]:

    grid = np.round(np.arange(w_min, w_max + 1e-9, step), 2)
    risks = np.array([risk(w, weeks_array) for w in grid])
    i = int(np.argmin(risks))
    w_star = float(grid[i])
    p_star = empirical_success_prob(weeks_array, w_star)
    return w_star, p_star, float(risks[i])

def excel_writer(path: str) -> Optional[pd.ExcelWriter]:
    eng = None
    try:
        import openpyxl
        eng = "openpyxl"
    except ImportError:
        try:
            import xlsxwriter
            eng = "xlsxwriter"
        except ImportError:
            eng = None
    if eng is None:
        return None
    return pd.ExcelWriter(path, engine=eng)

def main():

    df_raw = pd.read_excel(SRC_XLSX)
    df = df_raw.rename(columns={"孕妇BMI":"BMI", "检测孕期":"first_time"}).copy()
    df["weeks"] = to_weeks(df["first_time"])
    df = df[(df["weeks"]>=6) & (df["weeks"]<=35)][["BMI","weeks"]].dropna().reset_index(drop=True)


    labels = [f"Q{i}" for i in range(1,6)]
    df["BMI组"], _bins = pd.qcut(df["BMI"], q=5, labels=labels, retbins=True, duplicates="drop")

    # 基线最优 NIPT（各组）
    base_rows = []
    for lab in labels:
        sub = df[df["BMI组"]==lab]
        wk = sub["weeks"].to_numpy()
        w_star, p_star, r_star = find_optimal_week(wk)
        base_rows.append({
            "BMI组": lab,
            "人数": int(len(sub)),
            "BMI范围": f"[{sub['BMI'].min():.4f}, {sub['BMI'].max():.4f}]",
            "BMI中位数": float(sub["BMI"].median()),
            "首次达标周(中位数)": float(sub["weeks"].median()),
            "基线最优NIPT周数": round(w_star, 2),
            "基线该周达标概率": round(p_star, 3),
            "基线风险最小值": round(r_star, 3)
        })
    base_table = pd.DataFrame(base_rows).sort_values("BMI中位数").reset_index(drop=True)

    # 4) Bootstrap
    boot_rows = []
    rng = np.random.default_rng(42)
    for lab in labels:
        sub = df[df["BMI组"]==lab]
        wk = sub["weeks"].to_numpy()
        if len(wk) == 0:
            continue

        w_stars = []
        for _ in range(N_BOOT):
            idx = rng.integers(0, len(wk), len(wk))
            wk_b = wk[idx]
            w_star_b, _, _ = find_optimal_week(wk_b)
            w_stars.append(w_star_b)

        w_stars = np.array(w_stars)
        boot_rows.append({
            "BMI组": lab,
            "Bootstrap均值(最优周)": float(np.mean(w_stars)),
            "Bootstrap标准差": float(np.std(w_stars, ddof=1)),
            "CI2.5%(最优周)": float(np.percentile(w_stars, 2.5)),
            "CI97.5%(最优周)": float(np.percentile(w_stars, 97.5)),
            "样本量": int(len(wk)),
            "次数": int(N_BOOT)
        })
    boot_table = pd.DataFrame(boot_rows).sort_values("BMI组").reset_index(drop=True)

    # 保存结果
    os.makedirs(OUT_DIR, exist_ok=True)
    out_base_csv = os.path.join(OUT_DIR, "BMI五分位_基线分组与最优周.csv")
    out_boot_csv = os.path.join(OUT_DIR, "最优周_Bootstrap区间.csv")
    base_table.to_csv(out_base_csv, index=False)
    boot_table.to_csv(out_boot_csv, index=False)

    writer = excel_writer(os.path.join(OUT_DIR, "问题二_最优NIPT_Bootstrap.xlsx"))
    if writer is not None:
        with writer:
            base_table.to_excel(writer, sheet_name="基线最优NIPT", index=False)
            boot_table.to_excel(writer, sheet_name="Bootstrap_CI", index=False)

    print("【基线：各组最优 NIPT】")
    print(base_table[["BMI组","BMI范围","人数","基线最优NIPT周数","基线该周达标概率","基线风险最小值"]].to_string(index=False))
    print("\n【Bootstrap：最优周 95%CI】")
    print(boot_table[["BMI组","Bootstrap均值(最优周)","Bootstrap标准差","CI2.5%(最优周)","CI97.5%(最优周)"]].to_string(index=False))
    print("\nCSV 保存于：", out_base_csv, "与", out_boot_csv)

if __name__ == "__main__":
    main()
